# NYT + Guardian Archive Fetch

Bulk-fetch historical news headlines for use as economic event signals in the oil price world model.

**Sources:**
- **NYT Archive API** — one JSON dump per calendar month, all sections. Free key: https://developer.nytimes.com
- **Guardian Content API** — paginated search, finer section filters. Free key: https://open-platform.theguardian.com/access/

**Rate limits:**
- NYT: 500 requests/day, 5/min (we need only 24 requests for 2 years → fine)
- Guardian: 12/sec, 5000/day

**Output:** two parquet files (`nyt_2015_2016.parquet`, `guardian_2015_2016.parquet`) with normalized columns `date`, `headline`, `abstract`, `section`, `url`, `source`.

In [1]:
import os
import time
import json
import requests
import pandas as pd
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

# Load keys from .env in the project root (expects NYT_API_KEY and GUARDIAN_API_KEY).
load_dotenv()

NYT_API_KEY = os.environ['NYT_API_KEY']
GUARDIAN_API_KEY = os.environ.get('GUARDIAN_API_KEY', '')

OUT_DIR = Path('.')
OUT_DIR.mkdir(exist_ok=True)

YEARS = [2015]

## 1. NYT Archive API

Endpoint: `https://api.nytimes.com/svc/archive/v1/{year}/{month}.json?api-key=...`

Returns every article NYT published that month in a single response (typically 5–10k articles). We'll filter to economically relevant sections afterward.

In [2]:
def fetch_nyt_month(year: int, month: int, api_key: str) -> list[dict]:
    url = f'https://api.nytimes.com/svc/archive/v1/{year}/{month}.json'
    r = requests.get(url, params={'api-key': api_key}, timeout=60)
    r.raise_for_status()
    return r.json()['response']['docs']

In [8]:
# Sections / desks worth keeping for a macro-finance model.
# Inspect the sample above and adjust as needed.
NYT_KEEP_SECTIONS = {'Business Day', 'Business', 'World', 'U.S.'}
NYT_KEEP_DESKS = {
    'Business', 'Financial', 'Foreign', 'Washington',
    'Energy', 'Economy', 'National'
}

def normalize_nyt(doc: dict) -> dict | None:
    section = doc.get('section_name') or ''
    desk = doc.get('news_desk') or ''
    if section not in NYT_KEEP_SECTIONS or desk not in NYT_KEEP_DESKS:
        return None
    headline = (doc.get('headline') or {}).get('main') or ''
    if not headline:
        return None
    pub = doc.get('pub_date', '')[:10]
    return {
        'date': pub,
        'headline': headline,
        'abstract': doc.get('abstract') or '',
        'lead_paragraph': doc.get('lead_paragraph') or '',
        'snippet': doc.get('snippet') or '',
        'section': section or desk,
        'url': doc.get('web_url', ''),
        'source': 'nyt',
    }

nyt_rows = []
for year in YEARS:
    for month in range(1, 2):
        docs = fetch_nyt_month(year, month, NYT_API_KEY)
        kept = [r for r in (normalize_nyt(d) for d in docs) if r]
        nyt_rows.extend(kept)
        print(f'{year}-{month:02d}: {len(docs):5d} raw → {len(kept):4d} kept')
        time.sleep(12)  # stay under 5 req/min

nyt_df = pd.DataFrame(nyt_rows).sort_values('date').reset_index(drop=True)
nyt_df.to_parquet(OUT_DIR / 'nyt_2015_2016.parquet', index=False)
print(f'NYT total kept: {len(nyt_df)}')
nyt_df.head()

2015-01:  6906 raw → 1729 kept
NYT total kept: 1729


,date,headline,abstract,lead_paragraph,snippet,section,url,source
0,2015-01-01,Much of David Duke’s ’91 Campaign Is Now in Lo...,Representative Steve Scalise’s effort to expla...,"BATON ROUGE, La. — David Duke seems a figure f...",Representative Steve Scalise’s effort to expla...,U.S.,https://www.nytimes.com/2015/01/01/us/politics...,nyt
1,2015-01-01,With Schoolgirls Taken by Boko Haram Still Mis...,Distrust between the two countries has resurfa...,"STUTTGART, Germany — Soon after the Islamist g...",Distrust between the two countries has resurfa...,World,https://www.nytimes.com/2015/01/01/world/with-...,nyt
2,2015-01-01,"States’ Minimum Wages Rise, Helping Millions o...",Minimum wage increases go into effect in 20 st...,"For some low-wage workers, everyday tasks like...",Minimum wage increases go into effect in 20 st...,Business Day,https://www.nytimes.com/2015/01/01/business/ho...,nyt
3,2015-01-01,Euro and Immigration Promise Challenges for Me...,Though she is still popular at home and respec...,BERLIN — Although she is still popular at home...,Though she is still popular at home and respec...,World,https://www.nytimes.com/2015/01/01/world/euro-...,nyt
4,2015-01-01,Massachusetts: New Effort to Move Bombings Trial,"Lawyers for Dzhokhar Tsarnaev, the defendant i...","Lawyers for Dzhokhar Tsarnaev, the defendant i...","Lawyers for Dzhokhar Tsarnaev, the defendant i...",U.S.,https://www.nytimes.com/2015/01/01/us/massachu...,nyt


In [12]:
for i in range(105,125):
    print(f'NYT article {i+1}: \n\n')
    print(nyt_df.iloc[i]['date'])
    print(nyt_df.iloc[i]['headline'])
    print(nyt_df.iloc[i]['abstract'])
    print(nyt_df.iloc[i]['lead_paragraph'])
    print(nyt_df.iloc[i]['snippet'])
    print('\n\n')


NYT article 106: 


2015-01-04
Ulrich Beck, Sociologist Who Warned of Technology, Dies at 70
Mr. Beck shot to fame in 1986 with his book “Risk Society: Towards a New Modernity,” which argued that technology had also created new risks in spheres ranging from ecology to finance.
BERLIN — Ulrich Beck, a sociologist who became one of Germany’s most prominent public intellectuals by exploring the ways technology had created a new, riskier society, died on Thursday. He was 70.
Mr. Beck shot to fame in 1986 with his book “Risk Society: Towards a New Modernity,” which argued that technology had also created new risks in spheres ranging from ecology to finance.



NYT article 107: 


2015-01-04
December Auto Sales, and Job Data for U.S. and Europe
On Monday, automakers reported strong December and annual sales, and on Friday, the Labor Department will report on job creation and unemployment.

On Monday, automakers reported strong December and annual sales, and on Friday, the Labor Department wi

## 2. Guardian Content API

Endpoint: `https://content.guardianapis.com/search`

Paginated (max 200/page), filtered by `section` and date range. We grab the Business, World, and Politics sections.

In [ ]:
def fetch_guardian_section(section: str, from_date: str, to_date: str, api_key: str) -> list[dict]:
    url = 'https://content.guardianapis.com/search'
    all_results = []
    page = 1
    while True:
        params = {
            'section': section,
            'from-date': from_date,
            'to-date': to_date,
            'page-size': 200,
            'page': page,
            'show-fields': 'trailText,headline',
            'order-by': 'oldest',
            'api-key': api_key,
        }
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()['response']
        all_results.extend(data['results'])
        if page >= data['pages']:
            break
        page += 1
        time.sleep(0.1)  # well under 12/sec
    return all_results

def normalize_guardian(doc: dict) -> dict:
    fields = doc.get('fields') or {}
    return {
        'date': doc['webPublicationDate'][:10],
        'headline': fields.get('headline') or doc.get('webTitle', ''),
        'abstract': fields.get('trailText', ''),
        'section': doc.get('sectionName', ''),
        'url': doc.get('webUrl', ''),
        'source': 'guardian',
    }

GUARDIAN_SECTIONS = ['business', 'world', 'politics']

guardian_rows = []
for section in GUARDIAN_SECTIONS:
    docs = fetch_guardian_section(section, '2015-01-01', '2016-12-31', GUARDIAN_API_KEY)
    print(f'{section}: {len(docs)} articles')
    guardian_rows.extend(normalize_guardian(d) for d in docs)

guardian_df = pd.DataFrame(guardian_rows).sort_values('date').reset_index(drop=True)
guardian_df.to_parquet(OUT_DIR / 'guardian_2015_2016.parquet', index=False)
print(f'Guardian total: {len(guardian_df)}')
guardian_df.head()

## 3. Quick inspection

Sanity-check daily volume and a few sample headlines from oil-relevant dates.

In [ ]:
combined = pd.concat([nyt_df, guardian_df], ignore_index=True)
combined['date'] = pd.to_datetime(combined['date'])

daily_counts = combined.groupby(['date', 'source']).size().unstack(fill_value=0)
print(daily_counts.describe())
daily_counts.rolling(7).mean().plot(figsize=(12, 4), title='7-day rolling article count')

In [ ]:
# Example: headlines from the OPEC meeting week Nov 27 2014 aftermath / 2016 Feb oil bottom
for d in ['2015-01-05', '2015-12-04', '2016-02-11', '2016-09-28']:
    sub = combined[combined['date'] == d]
    print(f'\n=== {d} ({len(sub)} articles) ===')
    for _, row in sub.head(8).iterrows():
        print(f'  [{row["source"]:8s}] {row["headline"]}')